<a href="https://colab.research.google.com/github/gvessio/pensare-in-python/blob/main/notebook/capitolo_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capitolo 8 — Stringhe ed espressioni regolari

Traduzione italiana del capitolo 8 di [*Think Python*, terza edizione](https://allendowney.github.io/ThinkPython/chap08.html), di **Allen B. Downey** (© 2024). Testo distribuito con licenza [Creative Commons BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/); codice con licenza [MIT](https://mit-license.org/). Traduzione e adattamento didattico per il percorso di orientamento all'informatica del Dipartimento di Informatica dell'Università degli Studi di Bari Aldo Moro.

In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

Le stringhe sono diverse dagli interi, dai numeri in virgola mobile e dai booleani. Una stringa è una **sequenza**, cioè contiene più valori disposti in un ordine preciso. In questo capitolo vedremo come accedere ai valori che compongono una stringa e come usare funzioni che elaborano stringhe.

Utilizzeremo anche le espressioni regolari, uno strumento potente per individuare schemi ricorrenti nelle stringhe ed eseguire operazioni come la ricerca e la sostituzione.

Come esercizio, avrai l'occasione di applicare questi strumenti a un gioco di parole chiamato Wordle.

## Una stringa è una sequenza

Una stringa è una sequenza di caratteri. Un **carattere** può essere una lettera (di quasi qualsiasi alfabeto), una cifra, un segno di punteggiatura o uno spazio bianco.

Puoi selezionare un carattere di una stringa con l'operatore delle parentesi quadre. Questa istruzione seleziona il carattere numero 1 di `fruit` e lo assegna a `letter`:

In [ ]:
fruit = 'banana'
letter = fruit[1]

L'espressione tra parentesi quadre è un **indice** (*index*), chiamato così perché *indica* quale carattere della sequenza selezionare. Ma il risultato potrebbe sorprenderti.

In [ ]:
letter

La lettera con indice `1` è in realtà la seconda della stringa. Un indice indica uno spostamento rispetto all'inizio della stringa, quindi lo spostamento della prima lettera è `0`.

In [ ]:
fruit[0]

Puoi pensare a `'b'` come alla lettera in posizione zero di `'banana'`.

L'indice tra parentesi quadre può essere una variabile.

In [ ]:
i = 1
fruit[i]

Oppure un'espressione che contiene variabili e operatori.

In [ ]:
fruit[i+1]

Il valore dell'indice, però, deve essere un intero; altrimenti ottieni un `TypeError`.

In [ ]:
%%expect TypeError

fruit[1.5]

Come abbiamo visto nel Capitolo 1, possiamo usare la funzione predefinita `len` per ottenere la lunghezza di una stringa.

In [ ]:
n = len(fruit)
n

Per ottenere l'ultima lettera di una stringa, potresti essere tentato di scrivere:

In [ ]:
%%expect IndexError

fruit[n]

Ma questo provoca un `IndexError`, perché in `'banana'` non esiste una lettera con indice 6. Poiché abbiamo iniziato a contare da `0`, le sei lettere hanno indici da `0` a `5`. Per ottenere l'ultimo carattere devi sottrarre `1` da `n`:

In [ ]:
fruit[n-1]

C'è però un modo più semplice. Per ottenere l'ultima lettera di una stringa puoi usare un indice negativo, che conta all'indietro a partire dalla fine.

In [ ]:
fruit[-1]

L'indice `-1` seleziona l'ultima lettera, `-2` la penultima e così via.

## Porzioni di stringa (*slice*)

Un segmento di una stringa si chiama **slice**. Selezionare una porzione di stringa è simile a selezionare un carattere.

In [ ]:
fruit = 'banana'
fruit[0:3]

L'operatore `[n:m]` restituisce la parte della stringa che va dal carattere in posizione `n` a quello in posizione `m`, **includendo il primo ed escludendo il secondo**. Questo comportamento può sembrare controintuitivo, ma può essere utile immaginare che gli indici puntino agli spazi *tra* i caratteri, come nella figura:

In [ ]:
from diagram import make_binding, Element, Value

binding = make_binding("fruit", ' b a n a n a ')
elements = [Element(Value(i), None) for i in range(7)]

In [ ]:
import matplotlib.pyplot as plt
from diagram import diagram, adjust
from matplotlib.transforms import Bbox

width, height, x, y = [1.35, 0.54, 0.23, 0.39]

ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
bboxes = [bbox]

def draw_elts(x, y, elements):
    for elt in elements:
        bbox = elt.draw(ax, x, y, draw_value=False)
        bboxes.append(bbox)

        x1 = (bbox.xmin + bbox.xmax) / 2
        y1 = bbox.ymax + 0.02
        y2 = y1 + 0.14
        handle = plt.plot([x1, x1], [y1, y2], ':', lw=0.5, color='gray')
        x += 0.105

draw_elts(x + 0.48, y - 0.25, elements)
bbox = Bbox.union(bboxes)
# adjust(x, y, bbox)

Per esempio, la slice `[3:6]` seleziona le lettere `ana`: quindi `6` è valido come estremo di una slice, ma non come indice di un singolo carattere.

Se ometti il primo indice, la slice parte dall'inizio della stringa.

In [ ]:
fruit[:3]

Se ometti il secondo indice, la slice arriva fino alla fine della stringa:

In [ ]:
fruit[3:]

Se il primo indice è maggiore o uguale al secondo, il risultato è una **stringa vuota**, rappresentata da due virgolette:

In [ ]:
fruit[3:3]

Una stringa vuota non contiene caratteri e ha lunghezza 0.

Continuando con questo esempio, che cosa pensi significhi `fruit[:]`? Prova a eseguirlo.

In [ ]:
fruit[:]

## Le stringhe sono immutabili

Potresti essere tentato di usare l'operatore `[]` a sinistra di un'assegnazione per modificare un carattere di una stringa, così:

In [ ]:
%%expect TypeError

greeting = 'Hello, world!'
greeting[0] = 'J'

Il risultato è un `TypeError`. Nel messaggio di errore, l'«oggetto» è la stringa e l'«elemento» è il carattere che abbiamo provato ad assegnare.

L'errore si verifica perché le stringhe sono **immutabili**: non puoi modificare una stringa esistente. Puoi soltanto creare una nuova stringa che sia una variante di quella originale.

In [ ]:
new_greeting = 'J' + greeting[1:]
new_greeting

Questo esempio concatena una nuova lettera iniziale a una porzione di `greeting`. La stringa originale non viene modificata.

In [ ]:
greeting

## Confronto tra stringhe

Gli operatori relazionali funzionano anche con le stringhe. Per verificare se due stringhe sono uguali, possiamo usare l'operatore `==`.

In [ ]:
word = 'banana'

if word == 'banana':
    print('All right, banana.')

Altri operatori relazionali sono utili per ordinare le parole alfabeticamente:

In [ ]:
def compare_word(word):
    if word < 'banana':
        print(word, 'comes before banana.')
    elif word > 'banana':
        print(word, 'comes after banana.')
    else:
        print('All right, banana.')

In [ ]:
compare_word('apple')

Python non tratta maiuscole e minuscole come fanno le persone. Tutte le lettere maiuscole precedono quelle minuscole, quindi:

In [ ]:
compare_word('Pineapple')

Per risolvere questo problema, prima del confronto possiamo convertire le stringhe in un formato uniforme, per esempio tutto minuscolo.

## Metodi delle stringhe

Le stringhe mettono a disposizione **metodi** che eseguono diverse operazioni utili. Un metodo è simile a una funzione — accetta argomenti e restituisce un valore — ma ha una sintassi diversa. Per esempio, il metodo `upper` prende una stringa e restituisce una nuova stringa con tutte le lettere maiuscole.

Invece della sintassi di funzione `upper(word)`, si usa la sintassi di metodo `word.upper()`.

In [ ]:
word = 'banana'
new_word = word.upper()
new_word

Questo uso dell'operatore punto specifica il nome del metodo, `upper`, e quello della stringa a cui applicarlo, `word`. Le parentesi vuote indicano che questo metodo non richiede argomenti.

La chiamata di un metodo si chiama **invocazione** (*invocation*); in questo caso diciamo che invochiamo `upper` su `word`.

## Scrivere file

Gli operatori e i metodi delle stringhe sono utili per leggere e scrivere file di testo. Come esempio, lavoreremo sul testo di *Dracula*, romanzo di Bram Stoker disponibile su Project Gutenberg (<https://www.gutenberg.org/ebooks/345>).

In [ ]:
download('https://github.com/AllenDowney/ThinkPython/raw/v3/pg345.txt');

Il libro è stato scaricato in un file di testo semplice, `pg345.txt`, che possiamo aprire in lettura così:

L'argomento con nome `encoding='utf-8'` non compare nell'edizione cartacea; è necessario soltanto su Windows.

In [ ]:
reader = open('pg345.txt', encoding='utf-8')

Oltre al romanzo, il file contiene informazioni iniziali sul libro e informazioni finali sulla licenza. Possiamo eliminarle individuando le righe speciali che iniziano con `'***'`.

La funzione seguente verifica se una riga è speciale, usando il metodo `startswith`, che controlla se una stringa inizia con una certa sequenza di caratteri.

In [ ]:
def is_special_line(line):
    return line.startswith('*** ')

Usiamo la funzione per scorrere le righe del file e visualizzare soltanto quelle speciali.

In [ ]:
for line in reader:
    if is_special_line(line):
        print(line.strip())

Creiamo un file chiamato `pg345_cleaned.txt` che contenga soltanto il testo del romanzo. Per scorrerlo di nuovo dobbiamo riaprire il file originale in lettura e aprire quello nuovo in scrittura.

In [ ]:
reader = open('pg345.txt', encoding='utf-8')
writer = open('pg345_cleaned.txt', 'w', encoding='utf-8')

`open` accetta un parametro facoltativo che specifica la **modalità** (*mode*). Qui `'w'` significa scrittura: se il file non esiste viene creato; se esiste, il suo contenuto viene sostituito.

Per prima cosa scorriamo il file fino alla prima riga speciale.

In [ ]:
for line in reader:
    if is_special_line(line):
        break

L'istruzione `break` interrompe subito il ciclo, prima della fine del file.

Quando il ciclo termina, `line` contiene la riga speciale che ha reso vera la condizione.

In [ ]:
line

Poiché `reader` conserva la posizione raggiunta nel file, possiamo usare un secondo ciclo per riprendere da lì.

Il ciclo seguente legge il resto del file una riga alla volta. Se trova la riga speciale che segnala la fine del testo, si interrompe; altrimenti scrive la riga nel file di destinazione.

In [ ]:
for line in reader:
    if is_special_line(line):
        break
    writer.write(line)

Quando questo ciclo termina, `line` contiene la seconda riga speciale.

In [ ]:
line

Ora `reader` e `writer` sono ancora aperti: potremmo continuare a leggere e scrivere. Per indicare che abbiamo finito, chiudiamo entrambi invocando il metodo `close`.

In [ ]:
reader.close()
writer.close()

Per verificare che tutto abbia funzionato, leggiamo le prime righe del nuovo file.

In [ ]:
for line in open('pg345_cleaned.txt', encoding='utf-8'):
    line = line.strip()
    if len(line) > 0:
        print(line)
    if line.endswith('Stoker'):
        break

Il metodo `endswith` verifica se una stringa termina con una certa sequenza di caratteri.

## Ricerca e sostituzione

Nella traduzione islandese di *Dracula* del 1901, il nome di un personaggio fu cambiato da «Jonathan» a «Thomas». Per fare lo stesso nella versione inglese, possiamo scorrere il libro, usare `replace` per sostituire il nome e scrivere il risultato in un nuovo file.

Cominciamo contando le righe del file ripulito.

In [ ]:
total = 0
for line in open('pg345_cleaned.txt', encoding='utf-8'):
    total += 1

total

Per controllare se una riga contiene «Jonathan», possiamo usare `in`, che verifica se una sequenza di caratteri compare in un punto qualsiasi della riga.

In [ ]:
total = 0
for line in open('pg345_cleaned.txt', encoding='utf-8'):
    if 'Jonathan' in line:
        total += 1

total

Ci sono 199 righe che contengono il nome, ma il numero totale delle occorrenze è diverso, perché può comparire più volte nella stessa riga. Il metodo `count` conta quante volte una sequenza compare in una stringa.

In [ ]:
total = 0
for line in open('pg345_cleaned.txt', encoding='utf-8'):
    total += line.count('Jonathan')

total

Ora possiamo sostituire `'Jonathan'` con `'Thomas'` così:

In [ ]:
writer = open('pg345_replaced.txt', 'w', encoding='utf-8')

for line in open('pg345_cleaned.txt', encoding='utf-8'):
    line = line.replace('Jonathan', 'Thomas')
    writer.write(line)

Il risultato è un file chiamato `pg345_replaced.txt` che contiene una versione di *Dracula* in cui Jonathan Harker si chiama Thomas.

In [ ]:
total = 0
for line in open('pg345_replaced.txt', encoding='utf-8'):
    total += line.count('Thomas')

total

## Espressioni regolari

Se conosciamo esattamente la sequenza da cercare, possiamo usare `in` per trovarla e `replace` per sostituirla. Esiste però un altro strumento, chiamato **espressione regolare**, che può fare queste cose e molto altro.

Partiamo da un esempio semplice. Supponiamo di voler trovare tutte le righe che contengono una parola. Questa volta cerchiamo il personaggio che dà il titolo al libro, il conte Dracula. Ecco una riga che lo menziona.

In [ ]:
text = "I am Dracula; and I bid you welcome, Mr. Harker, to my house."

Ed ecco il **pattern** (schema di ricerca) che useremo.

In [ ]:
pattern = 'Dracula'

Il modulo `re` fornisce funzioni per le espressioni regolari. Possiamo importarlo e usare `search` per verificare se il pattern compare nel testo.

In [ ]:
import re

result = re.search(pattern, text)
result

Se il pattern compare nel testo, `search` restituisce un oggetto `Match` con i risultati della ricerca. Tra le altre informazioni, l'attributo `string` contiene il testo esaminato.

In [ ]:
result.string

Il metodo `group` restituisce la parte di testo che corrisponde al pattern.

In [ ]:
result.group()

Il metodo `span` restituisce gli indici in cui la corrispondenza inizia e termina.

In [ ]:
result.span()

Se il pattern non compare nel testo, `search` restituisce `None`.

In [ ]:
result = re.search('Count', text)
print(result)

Possiamo quindi verificare se la ricerca è riuscita controllando se il risultato è `None`.

In [ ]:
result == None

Mettendo insieme questi elementi, ecco una funzione che scorre le righe del romanzo finché non ne trova una corrispondente al pattern e restituisce l'oggetto `Match`.

In [ ]:
def find_first(pattern):
    for line in open('pg345_cleaned.txt', encoding='utf-8'):
        result = re.search(pattern, line)
        if result != None:
            return result

Possiamo usarla per trovare la prima menzione di un personaggio.

In [ ]:
result = find_first('Harker')
result.string

Per questo esempio avremmo potuto usare più facilmente l'operatore `in`. Ma le espressioni regolari possono fare cose che `in` non può fare.

Per esempio, la barra verticale `'|'` consente di cercare la sequenza a sinistra **oppure** quella a destra. Supponiamo di cercare la prima menzione di Mina Murray senza sapere se viene indicata con il nome o con il cognome. Il pattern seguente riconosce entrambi.

In [ ]:
pattern = 'Mina|Murray'
result = find_first(pattern)
result.string

Un pattern del genere permette anche di contare quante volte un personaggio viene menzionato con uno dei due nomi. Ecco una funzione che conta le righe corrispondenti al pattern.

In [ ]:
def count_matches(pattern):
    count = 0
    for line in open('pg345_cleaned.txt', encoding='utf-8'):
        result = re.search(pattern, line)
        if result != None:
            count += 1
    return count

Vediamo quante volte viene menzionata Mina.

In [ ]:
count_matches('Mina|Murray')

Il carattere speciale `'^'` corrisponde all'inizio di una stringa, quindi possiamo trovare una riga che inizia con un certo pattern.

In [ ]:
result = find_first('^Dracula')
result.string

Il carattere speciale `'$'` corrisponde alla fine di una stringa, quindi possiamo trovare una riga che termina con un certo pattern (ignorando il carattere di nuova riga finale).

In [ ]:
result = find_first('Harker$')
result.string

## Sostituzione nelle stringhe

Bram Stoker nacque in Irlanda e viveva in Inghilterra quando *Dracula* fu pubblicato nel 1897. Ci aspettiamo quindi che usasse la grafia britannica di parole come «centre» e «colour». Il pattern seguente riconosce sia «centre» sia la grafia americana «center».

In [ ]:
pattern = 'cent(er|re)'

Le parentesi racchiudono la parte del pattern a cui si applica la barra verticale. Il pattern riconosce una sequenza che inizia con `'cent'` e termina con `'er'` oppure `'re'`.

In [ ]:
result = find_first(pattern)
result.string

Come previsto, Stoker usava la grafia britannica.

Controlliamo anche «colour». Il pattern seguente usa il carattere speciale `'?'`, che rende facoltativo il carattere precedente.

In [ ]:
pattern = 'colou?r'

Il pattern riconosce «colour» con la `'u'` e «color» senza di essa.

In [ ]:
result = find_first(pattern)
line = result.string
line

Ancora una volta, come previsto, Stoker usava la grafia britannica.

Per produrre un'edizione con la grafia americana possiamo usare la funzione `sub` del modulo `re`, che esegue una **sostituzione di stringhe**.

In [ ]:
re.sub(pattern, 'color', line)

Il primo argomento è il pattern da cercare, il secondo è il testo sostitutivo e il terzo è la stringa in cui cercare. Nel risultato «colour» è stato sostituito con «color».

In [ ]:
# Ho usato questa funzione per cercare righe da utilizzare come esempi

def all_matches(pattern):
    for line in open('pg345_cleaned.txt', encoding='utf-8'):
        result = re.search(pattern, line)
        if result:
            print(line.strip())

In [ ]:
# Ecco il pattern usato (contiene alcune funzionalità che non abbiamo ancora visto)

names = r'(?<!\.\s)[A-Z][a-zA-Z]+'

all_matches(names)

## Debugging

Quando si leggono e scrivono file, il debugging può essere complicato. In un notebook Jupyter possiamo usare i **comandi della shell**. Per visualizzare le prime righe di un file, per esempio, usiamo `!head`:

In [ ]:
!head pg345_cleaned.txt

Il punto esclamativo `!` indica un comando della shell, non un'istruzione Python. Per visualizzare le ultime righe usiamo `!tail`.

In [ ]:
!tail pg345_cleaned.txt

Con file molto grandi, l'output può essere troppo lungo per controllarlo a mano. Una buona strategia di debugging è iniziare con una piccola parte del file, far funzionare il programma e poi usare il file intero.

Per creare un file piccolo a partire da uno più grande, possiamo usare `!head` con l'operatore di reindirizzamento `>`, che scrive il risultato in un file anziché visualizzarlo.

In [ ]:
!head pg345_cleaned.txt > pg345_cleaned_10_lines.txt

Per impostazione predefinita `!head` legge le prime 10 righe, ma accetta un argomento facoltativo per specificare quante righe leggere.

In [ ]:
!head -100 pg345_cleaned.txt > pg345_cleaned_100_lines.txt

Questo comando legge le prime 100 righe di `pg345_cleaned.txt` e le scrive in `pg345_cleaned_100_lines.txt`.

**Nota:** `!head` e `!tail` non sono disponibili su tutti i sistemi operativi. Se non funzionano, possiamo scrivere funzioni equivalenti in Python. Vedi il primo esercizio.

## Glossario

**sequenza (*sequence*):** raccolta ordinata di valori, ciascuno identificato da un indice intero.

**carattere (*character*):** elemento di una stringa, compresi lettere, numeri e simboli.

**indice (*index*):** intero usato per selezionare un elemento di una sequenza. In Python gli indici partono da `0`.

**slice:** parte di una stringa individuata da un intervallo di indici.

**stringa vuota (*empty string*):** stringa senza caratteri, di lunghezza `0`.

**oggetto (*object*):** qualcosa a cui una variabile può fare riferimento; ha un tipo e un valore.

**immutabile (*immutable*):** oggetto i cui elementi non possono essere modificati.

**invocazione (*invocation*):** espressione, o parte di essa, che chiama un metodo.

**espressione regolare (*regular expression*):** sequenza di caratteri che definisce uno schema di ricerca.

**pattern:** regola che definisce i requisiti per ottenere una corrispondenza.

**sostituzione di stringhe (*string substitution*):** sostituzione di una stringa o parte di essa con un'altra.

**comando della shell (*shell command*):** istruzione in un linguaggio usato per interagire con il sistema operativo.

## Esercizi

In [ ]:
# Questa cella attiva informazioni di debugging dettagliate in Jupyter
# quando si verifica un errore durante l’esecuzione. Eseguila prima degli esercizi.

%xmode Verbose

In [ ]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

### Chiedi a un assistente virtuale

Abbiamo appena sfiorato le possibilità delle espressioni regolari. Per approfondire, chiedi: «Quali sono i caratteri speciali più comuni nelle espressioni regolari di Python?».

Puoi anche chiedere pattern che riconoscano particolari stringhe:

* Un numero di telefono di 10 cifre con trattini.
* Un indirizzo stradale con numero civico e nome della via, seguito da `ST` o `AVE`.
* Un nome completo preceduto da un titolo come `Mr` o `Mrs`, seguito da un numero qualsiasi di nomi con iniziale maiuscola, eventualmente separati da trattini.

Come sfida più complessa, chiedi un'espressione regolare che riconosca qualsiasi URL valido.

Le espressioni regolari sono spesso precedute da `r` prima delle virgolette, che indica una *raw string*. Chiedi: «Che cos'è una raw string in Python?».

In [ ]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

### Esercizio

Scrivi una funzione equivalente al comando della shell `!head`. Deve ricevere il nome di un file da leggere, il numero di righe da leggere e il nome del file in cui scriverle. Se il terzo parametro è `None`, deve visualizzare le righe invece di scriverle.

Puoi chiedere aiuto a un assistente virtuale, ma specifica di non usare né `with` né `try`.

In [ ]:
# Scrivi qui la soluzione

Puoi usare gli esempi seguenti per verificare la tua funzione.

In [ ]:
head('pg345_cleaned.txt', 10)

In [ ]:
head('pg345_cleaned.txt', 100, 'pg345_cleaned_100_lines.txt')

In [ ]:
!tail pg345_cleaned_100_lines.txt

### Esercizio

«Wordle» è un gioco online in cui bisogna indovinare una parola di cinque lettere in sei tentativi o meno. Ogni tentativo deve essere una parola riconosciuta, esclusi i nomi propri. Dopo ogni tentativo scopri quali lettere compaiono nella soluzione e quali sono nella posizione corretta.

Per esempio, se la soluzione è `MOWER` e proponi `TRIED`, scopri che `E` è presente nella posizione giusta, `R` è presente ma nella posizione sbagliata, mentre `T`, `I` e `D` non compaiono.

Supponi invece di aver proposto `SPADE` e `CLERK` e di aver scoperto che `E` è presente, ma non in nessuna delle posizioni provate, e che nessuna delle altre lettere compare. Quante parole dell'elenco potrebbero essere la soluzione? Scrivi una funzione `check_word` che riceva una parola di cinque lettere e verifichi se è compatibile con questi tentativi.

In [ ]:
# Scrivi qui la soluzione

Puoi usare una qualsiasi delle funzioni del capitolo precedente, come `uses_any`.

In [ ]:
def uses_any(word, letters):
    for letter in word.lower():
        if letter in letters.lower():
            return True
    return False

Puoi usare il ciclo seguente per verificare la tua funzione.

In [ ]:
for line in open('words.txt'):
    word = line.strip()
    if len(word) == 5 and check_word(word):
        print(word)

### Esercizio

Proseguendo l'esercizio precedente, supponi di proporre `TOTEM` e scoprire che la `E` *non* è ancora nella posizione giusta, mentre la `M` sì. Quante parole rimangono?

In [ ]:
# Scrivi qui la soluzione

In [ ]:
# Scrivi qui la soluzione

### Esercizio

*Il conte di Montecristo* è un romanzo di Alexandre Dumas considerato un classico. Eppure, nell'introduzione a una traduzione inglese, Umberto Eco confessa di averlo trovato «uno dei romanzi scritti peggio di tutti i tempi».

In particolare, sostiene che ripeta spudoratamente gli stessi aggettivi e cita quante volte i personaggi rabbrividiscono o impallidiscono.

Per verificare questa osservazione, conta le righe che contengono la parola inglese `pale` in una qualsiasi delle sue forme, tra cui `pale`, `pales`, `paled` e `paleness`, oltre alla parola collegata `pallor`. Usa un'unica espressione regolare. Come sfida aggiuntiva, evita corrispondenze con parole diverse, come `impale`: puoi chiedere aiuto a un assistente virtuale.

La cella seguente scarica il libro da Project Gutenberg: <https://www.gutenberg.org/ebooks/1184>.

In [ ]:
download('https://github.com/AllenDowney/ThinkPython/raw/v3/pg1184.txt');

La cella seguente esegue una funzione che legge il file di Project Gutenberg e scrive un nuovo file con il solo testo del romanzo, senza le informazioni aggiuntive.

In [ ]:
def clean_file(input_file, output_file):
    reader = open(input_file, encoding='utf-8')
    writer = open(output_file, 'w', encoding='utf-8')

    for line in reader:
        if is_special_line(line):
            break

    for line in reader:
        if is_special_line(line):
            break
        writer.write(line)

    reader.close()
    writer.close()

clean_file('pg1184.txt', 'pg1184_cleaned.txt')

In [ ]:
# Scrivi qui la soluzione

In [ ]:
# Scrivi qui la soluzione

In [ ]:
# Scrivi qui la soluzione

Secondo questo conteggio, queste parole compaiono in `223` righe del libro: forse Eco non aveva tutti i torti.